# Pilot XAI-DSS trên IndPenSim — bản v4 (sửa điểm mù của lời giải thích)

So với v3:
1. **Bổ sung dữ liệu có chủ đích.** Cho DSS chạy trên các mẻ *huấn luyện*; tại mỗi khuyến nghị, mô phỏng đủ 16 tổ hợp thay đổi rồi thêm vào dữ liệu. Lặp 2 vòng. Các mẻ kiểm thử không bị đụng tới.
2. **Ghi lượng khí sục** của mọi chính sách, báo theo % so với công thức.
3. **Đường cong học**: 60 / 120 / 180 / 240 mẻ dữ liệu ngẫu nhiên, và 240 mẻ + dữ liệu bổ sung.

Dữ liệu gốc giống hệt v3 (cùng seed), nên kết quả v3 chính là mốc "trước khi sửa".

**Trước khi chạy:** upload `pilot_colab_v4.zip` và notebook này vào `MyDrive/De_Tai_Nong_San/`, chọn *Runtime → Change runtime type → T4 GPU*, rồi chạy lần lượt từng ô.

## Ô 1. Kết nối Google Drive, giải nén code vào `pilot_v4/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
DRIVE_DIR = '/content/drive/MyDrive/De_Tai_Nong_San'   # sửa nếu bạn để file ở thư mục khác
ZIP_PATH = os.path.join(DRIVE_DIR, 'pilot_colab_v4.zip')
WORK = os.path.join(DRIVE_DIR, 'pilot_v4')
V2_DIR = os.path.join(DRIVE_DIR, 'pilot_v2', 'runs', 'v2')

if not os.path.exists(os.path.join(WORK, 'augment_gpu.py')):
    zipfile.ZipFile(ZIP_PATH).extractall(DRIVE_DIR)
    print('Đã giải nén vào', WORK)
os.chdir(WORK)
os.environ['PILOT_EXP'] = 'v4'      # kết quả ghi vào runs/v4/
print('Thư mục làm việc:', os.getcwd())

## Ô 2. Kiểm tra GPU và thư viện

In [ ]:
!nvidia-smi -L
!pip install -q lightgbm shap
import jax, lightgbm, shap
print('JAX', jax.__version__, '| thiết bị:', jax.devices())

## Ô 3 (có thể bỏ qua). Kiểm chứng lại bộ mô phỏng GPU

Bộ mô phỏng không đổi so với v3, chỉ ghi thêm lượng khí sục. Chạy ô này nếu muốn chắc chắn; dòng cuối phải là "khớp".

In [ ]:
!python -W ignore validate_gpu.py --v2-dir "{V2_DIR}"

## Ô 4. Quy mô (giống v3 để so sánh)

In [ ]:
N_TRAIN, N_TEST, ROUNDS = 240, 48, 2
print('Mẻ huấn luyện:', N_TRAIN, '| mẻ kiểm thử:', N_TEST, '| số vòng bổ sung dữ liệu:', ROUNDS)

## Ô 5. Dữ liệu gốc (hành động ngẫu nhiên) và mô hình ban đầu — giống v3

In [ ]:
!python -W ignore gen_data_gpu.py --seeds {N_TRAIN}
!python train_dss.py

## Ô 6. Bổ sung dữ liệu quanh khuyến nghị của DSS và huấn luyện lại (2 vòng)

Mỗi vòng: DSS chạy trên 240 mẻ huấn luyện; tại mỗi mốc quyết định mô phỏng 16 tổ hợp thay đổi (tối đa 3.840 nhánh song song); sau đó huấn luyện lại mô hình.

In [ ]:
for r in range(1, ROUNDS + 1):
    !python -W ignore augment_gpu.py --round {r} --seeds {N_TRAIN}
    !python train_dss.py

## Ô 7. Đánh giá 4 chính sách trên 48 mẻ kiểm thử và kiểm chứng lời giải thích (GPU)

In [ ]:
!python -W ignore eval_dss_gpu.py --seeds {N_TEST}

## Ô 8. Tổng hợp số liệu và đường cong học

In [ ]:
!python analyze.py
!python learning_curve.py

## Ô 9. Gom kết quả thành `ket_qua_pilot_v4.zip` trên Drive

Tải file này về, đặt vào `D:\De_Tai_Nong_San\`, rồi báo Claude để phân tích.

In [ ]:
out = os.path.join(DRIVE_DIR, 'ket_qua_pilot_v4.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ['runs/v4/results', 'runs/v4/models']:
        for root, _, files in os.walk(folder):
            for f in files:
                z.write(os.path.join(root, f))
    if os.path.exists('runs/v4/data/train.csv'):
        z.write('runs/v4/data/train.csv')
print('Đã lưu', out, f'({os.path.getsize(out)/1e6:.1f} MB)')